# Machine Learning II - Project 4
## Composing a Bach Chorale 
- In this project, taken from Geron, you will use CNNs and RNNs to create a Bach chorale
- It is composed of 382 chorales composed by Johann Sebastian Bach
- Each chorale is 100 to 640 time steps long, and each time step contains four integers
- Each of these integers corresponds to a note’s index on a piano, with zero indicating that no note is played
- Train a model consisting of both RNN and CNN layers that can predict the next time step (four notes), given a sequence of time steps from a chorale
- It does not matter to me if you choose to utilize the basic (vanilla) RNN, an LSTM, or a GRU
- Use your model to generate “Bach-like” music, one note at a time
- This can be accomplished by giving the model the start of a chorale and letting it compose, i.e., predict, the next time step
- Then, append this result to the input sequence and have the model compose the subsequent note, and so on.

In [1]:
from pathlib import Path
import tarfile
import urllib.request
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import time
import torch.nn.functional as F
from torch.utils.data import DataLoader
import random

from torch.utils.data.sampler import SubsetRandomSampler

In [2]:
data_directory = "jsb_chorales"
file_path = Path(data_directory) / 'train' / 'chorale_000.csv'
print(file_path.read_text()[:84])

note0,note1,note2,note3
74,70,65,58
74,70,65,58
74,70,65,58
74,70,65,58
75,70,58,55



In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(1901)
np.random.seed(1901)
print(f"Using device: {device}")

Using device: cuda


In [4]:
import csv

def load_chorales(data_directory):
    chorales = {"train": [], "valid": [], "test": []}
    for subset, chorale_list in chorales.items():
        for chorale_path in sorted((data_directory / subset).glob("*.csv")):
            with chorale_path.open() as f:
                reader = csv.reader(f)
                next(reader)  # skip header row
                chorale = [[int(note) for note in row] for row in reader]
                chorale_list.append(torch.tensor(chorale))
    return chorales

chorales = load_chorales(Path(data_directory))

In [ ]:
chorales["train"][0][:5]

In [5]:
def preprocess_note_indices(note_indices, length=None):
    notes_one_hot = F.one_hot(note_indices, num_classes=48).to(torch.float32)
    if length is None:
        length = len(note_indices)
    idx = torch.arange(len(note_indices), dtype=torch.float32,
                       device=note_indices.device).view(-1, 1) - 1
    return torch.cat([
        notes_one_hot,
        idx % 4,
        idx % 16 // 4,
        idx / (length - 1)
    ], dim=-1)

class BachDataset(torch.utils.data.Dataset):
    def __init__(self, chorales, window_length):
        self.chorales = chorales
        self.windows = []
        sos = torch.tensor([1])
        for chorale in chorales:
            notes = chorale.view(-1)
            note_indices = torch.cat([sos, F.relu(notes - 34)])
            X = preprocess_note_indices(note_indices)
            for index in range(len(notes) - window_length):
                self.windows.append((
                    X[index : index + window_length],
                    note_indices[index + 1 : index + window_length + 1]
                ))

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, index):
        return self.windows[index]

window_length = 128
train_set = BachDataset(chorales["train"], window_length)
valid_set = BachDataset(chorales["valid"], window_length)
test_set = BachDataset(chorales["test"], window_length)

In [6]:
torch.manual_seed(42)
batch_size = 32
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
valid_loader = DataLoader(valid_set, batch_size=batch_size)
test_loader = DataLoader(test_set, batch_size=batch_size)

data_loaders = {'train' : train_loader, 'valid': valid_loader, 'test' : test_loader}

In [ ]:
n_workers = 2
batch_size = 32 # training batch size
valid_size = 0.2 # proportion of downloaded training data to use for validation

data_loaders = get_data_loaders(train_set, valid_set, test_set, batch_size, valid_size, n_workers)

In [ ]:
class CharGRU(nn.Module):
    def __init__(self, input_dim=51, hidden_dim=256, num_layers=2, dropout=0.3, vocab_size=48):
        super().__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, num_layers, 
                          batch_first=True, 
                          dropout=dropout if num_layers > 1 else 0)
        self.dropout = nn.Dropout(dropout)
        self.output = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden=None):
        gru_out, hidden = self.gru(x, hidden)
        output = self.dropout(gru_out)
        output = self.output(output)
        return output, hidden

In [25]:
class HybridGRUModel(nn.Module):
    def __init__(self, input_dim=51, hidden_dim=256, num_layers=2, dropout=0.3, vocab_size=48):
        super(HybridGRUModel, self).__init__()
        self.conv1d = CausalConv1d(input_dim, 64, kernel_size=2, dilation=2**1)
        self.relu = nn.ReLU()
        self.gru = nn.GRU(64, hidden_dim, num_layers,
                          batch_first=True,
                          dropout=dropout if num_layers > 1 else 0)
        self.dropout = nn.Dropout(dropout)
        self.output = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        x = x.transpose(1, 2)
        x = self.relu(self.conv1d(x))
        x = x.transpose(1, 2)
        x, hidden = self.gru(x)
        x = self.dropout(x)
        x = self.output(x)
        return x.transpose(1, 2)

In [7]:
class CausalConv1d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dilation=1, **kwargs):
        super().__init__()
        self.pad = (kernel_size - 1) * dilation
        self.conv = nn.Conv1d(in_channels, out_channels, kernel_size, padding=self.pad, dilation=dilation, **kwargs)

    def forward(self, x):
        out = self.conv(x)
        return out[:, :, :-self.pad] if self.pad > 0 else out

In [ ]:


class BachModel(nn.Module):
    def __init__(self, n_inputs=51, conv_dim=32, lstm_dim=64, n_notes=48):
        super().__init__()
        conv_layers = []
        for idx in range(4):
            conv_layers += [
                CausalConv1d(n_inputs, conv_dim, kernel_size=2, dilation=2**idx),
                nn.ReLU(),
                nn.BatchNorm1d(conv_dim),
            ]
            n_inputs = conv_dim

        self.conv_stack = nn.Sequential(*conv_layers)
        self.lstm = nn.LSTM(input_size=conv_dim, hidden_size=lstm_dim, batch_first=True)
        self.output = nn.Linear(lstm_dim, n_notes)

    def forward(self, X):
        Z = X.transpose(1, 2)
        Z = self.conv_stack(Z)
        Z = Z.transpose(1, 2)
        Z, _ = self.lstm(Z)
        Z = self.output(Z)
        return Z.transpose(1, 2)

torch.manual_seed(42)
model = BachModel().to(device)


In [9]:
import torchmetrics

In [10]:
def evaluate_tm(model, data_loader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)
    return metric.compute()

def train(model, optimizer, loss_fn, metric, train_loader, valid_loader,
          n_epochs, patience=10, factor=0.1):
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", patience=patience, factor=factor)
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
    for epoch in range(n_epochs):
        total_loss = 0.0
        metric.reset()
        model.train()
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = loss_fn(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            metric.update(y_pred, y_batch)
        history["train_losses"].append(total_loss / len(train_loader))
        history["train_metrics"].append(metric.compute().item())
        val_metric = evaluate_tm(model, valid_loader, metric).item()
        history["valid_metrics"].append(val_metric)
        scheduler.step(val_metric)
        print(f"Epoch {epoch + 1}/{n_epochs}, "
              f"train loss: {history['train_losses'][-1]:.4f}, "
              f"train metric: {history['train_metrics'][-1]:.4f}, "
              f"valid metric: {history['valid_metrics'][-1]:.4f}")
    return history


In [26]:
model = HybridGRUModel().to(device)

In [27]:
torch.manual_seed(42)
optimizer = torch.optim.NAdam(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=48).to(device)
history = train(model, optimizer, criterion, accuracy, train_loader,
                valid_loader, n_epochs=5)

Epoch 1/5, train loss: 1.1943, train metric: 0.6523, valid metric: 0.8177
Epoch 2/5, train loss: 0.5608, train metric: 0.8319, valid metric: 0.8338
Epoch 3/5, train loss: 0.4730, train metric: 0.8540, valid metric: 0.8350
Epoch 4/5, train loss: 0.4160, train metric: 0.8698, valid metric: 0.8331
Epoch 5/5, train loss: 0.3712, train metric: 0.8830, valid metric: 0.8301


In [28]:
evaluate_tm(model, test_loader, accuracy)

tensor(0.8296, device='cuda:0')

In [29]:
# Generate a completely random sequence and see how it does
random_input = torch.randint(0, 48, (1, 128)).to(device)
random_X = preprocess_note_indices(random_input[0]).unsqueeze(0).to(device)

model.eval()
with torch.no_grad():
    output = model(random_X)
    predicted = output.argmax(dim=1)
    print(predicted)

tensor([[35, 28, 18, 12, 25, 36, 28, 21, 42, 39, 42, 26, 30, 23, 39, 27, 36, 24,
         25, 45, 30, 30, 14, 30, 23, 23, 19, 31, 40, 36, 35, 42, 38, 23, 18, 37,
         38, 38, 17, 43, 40, 36,  9, 18, 27, 35, 19,  7, 40, 36, 28, 18, 43, 38,
         19, 14, 45, 29, 24, 32, 16, 41, 35, 33, 45, 38,  0, 23, 46, 31, 14, 14,
         33, 21, 21, 21, 41, 24, 21, 21, 30, 40, 30, 17, 45, 38, 11, 42, 40, 28,
         42, 24, 28, 29, 23, 40, 40,  0, 40, 28, 42, 43, 12, 40, 28, 16, 28, 32,
         41, 38, 38, 23, 28, 27, 23,  6, 13, 26, 29, 17, 42, 34, 31, 14, 42, 36,
         24, 38]], device='cuda:0')


In [ ]:
# Recreate model, optimizer, and scheduler
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3)
loss = nn.CrossEntropyLoss()

In [ ]:
# Count the number of parameters for each model
def count_parameters(model):
    """Count trainable parameters"""
    return sum(p.numel() for p in model.parameters())


print(f"GRU:  {count_parameters(model):,}")

In [ ]:
def train_one_epoch(train_dataloader, model, optimizer, loss):
    """
    Performs one epoch of training
    """
    if torch.cuda.is_available():
        model.cuda()  # -
    model.train()
    train_loss = 0.0

    for batch_idx, (data, target) in tqdm(
        enumerate(train_dataloader),
        desc="Training",
        total=len(train_dataloader),
        leave=True,
        ncols=80,
    ):
        if torch.cuda.is_available():
            data, target = data.cuda(), target.cuda()
        optimizer.zero_grad()
        output = model(data)
        loss_value = loss(output, target)
        loss_value.backward() 
        optimizer.step()
        train_loss = train_loss + (
            (1 / (batch_idx + 1)) * (loss_value.data.item() - train_loss))

    return train_loss

In [ ]:
def valid_one_epoch(valid_dataloader, model, loss):
    """
    Validate at the end of one epoch
    """

    with torch.no_grad():
        model.eval()
        if torch.cuda.is_available():
            model.cuda()
        valid_loss = 0.0
        for batch_idx, (data, target) in tqdm(
            enumerate(valid_dataloader),
            desc="Validating",
            total=len(valid_dataloader),
            leave=True,
            ncols=80,
        ):
            if torch.cuda.is_available():
                data, target = data.cuda(), target.cuda()
            output = model(data)
            loss_value = loss(output, target)
            valid_loss = valid_loss + (
                (1 / (batch_idx + 1)) * (loss_value.data.item() - valid_loss))

    return valid_loss

In [ ]:
def optimize(data_loaders, model, optimizer, loss, n_epochs, save_path, patience, scheduler, interactive_tracking=False):
    if interactive_tracking:
        liveloss = PlotLosses()
    else:
        liveloss = None
    valid_loss_min = None
    logs = {}
    wait = 0

    for epoch in range(1, n_epochs + 1):

        train_loss = train_one_epoch(
            data_loaders["train"], model, optimizer, loss
        )

        valid_loss = valid_one_epoch(data_loaders["valid"], model, loss)
        print(f"Epoch: {epoch} \tTraining Loss: {train_loss:.6f} \tValidation Loss: {valid_loss:.6f}")
        if valid_loss_min is None or ((valid_loss_min - valid_loss) / valid_loss_min > 0.01):
            print(f"New minimum validation loss: {valid_loss:.6f}. Saving model ...")

            # Save the weights to save_path
            torch.save(model.state_dict(), save_path)  # -

            valid_loss_min = valid_loss
            wait = 0
        else:
            wait +=1
            if wait >= patience:
                print("Early stopping!")
                break
            
        scheduler.step(valid_loss)

        if interactive_tracking:
            logs["loss"] = train_loss
            logs["val_loss"] = valid_loss
            logs["lr"] = optimizer.param_groups[0]["lr"]
            liveloss.update(logs)
            liveloss.send()

In [ ]:
from livelossplot import PlotLosses
from livelossplot.outputs import MatplotlibPlot
from tqdm import tqdm

In [ ]:
n_epochs = 100

optimize(
    data_loaders,
    model,
    optimizer,
    loss,
    n_epochs,
    'baseline.pt',
    7,
    scheduler,
    interactive_tracking=True
)

In [ ]:
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=48).to(device)
evaluate_tm(model, test_loader, accuracy)